# N12 — Overtake Probability Model

**LightGBM binary classifier** — `P(overtake | lap features)`

Trained on the labeled pair dataset produced by N11. Predicts whether car X overtakes car Y **on the current lap**, given the observable state of the pair at lap boundary.

| | |
|---|---|
| Input | `data/processed/overtake_labeled/overtake_pairs_2023_2025.parquet` |
| Train | 2023 + 2024 |
| Test | 2025 |
| Export | `data/models/overtake_probability/` |

### Model choice — LightGBM

This notebook trains a **gradient boosted decision tree** classifier using [LightGBM](https://lightgbm.readthedocs.io/) (Light Gradient Boosting Machine), developed by Microsoft Research ([Ke et al., 2017](https://proceedings.neurips.cc/paper_files/paper/2017/file/6449f44a102fde848669bdd9eb6b76fa-Paper.pdf)).

LightGBM extends classical gradient boosting with two key algorithmic improvements:

- **GOSS (Gradient-based One-Side Sampling):** keeps all samples with large gradients (high error) and randomly drops low-gradient ones, reducing training data without significant information loss.
- **EFB (Exclusive Feature Bundling):** merges mutually exclusive sparse features into single bundles, cutting the effective feature dimensionality.

The result is significantly faster training than XGBoost or standard GBDT with comparable or better accuracy on tabular data.

**Why LightGBM for overtake prediction:**
- Native support for categorical features (`compound_x`, `compound_y`, `circuit_cluster`) without one-hot encoding — it uses optimal histogram splits directly.
- Built-in class imbalance handling via `is_unbalance=True`, avoiding manual oversampling.
- Probabilistic output (`predict_proba`) that the Strategy Agent can consume directly as `P(overtake)`.
- Fast inference — critical when the agent scores dozens of car pairs per simulated lap.


---

## Step 0 — Setup

Standard imports plus LightGBM and SHAP. Paths follow the same convention as N11: `repo_root` resolved via `.git` walker, outputs split between the notebook's `outputs/` folder and `data/models/` for exportable artifacts.


In [ ]:
# ── Step 0 · Setup ────────────────────────────────────────────────────────────
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report, log_loss
)
from sklearn.calibration import calibration_curve
import shap


In [9]:
# ── repo root ─────────────────────────────────────────────────────────────────
repo_root = Path().resolve()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── paths ─────────────────────────────────────────────────────────────────────
OUTPUTS    = repo_root / "notebooks/strategy/overtake_probability/outputs"
PROCESSED  = repo_root / "data/processed/overtake_labeled"
EXPORT_DIR = repo_root / "data/models/overtake_probability"

OUTPUTS.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("repo_root  :", repo_root)
print("OUTPUTS    :", OUTPUTS)
print("PROCESSED  :", PROCESSED)
print("EXPORT_DIR :", EXPORT_DIR)
print("\nlightgbm   :", lgb.__version__)
print("shap       :", shap.__version__)

repo_root  : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
OUTPUTS    : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\strategy\overtake_probability\outputs
PROCESSED  : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\overtake_labeled
EXPORT_DIR : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\overtake_probability

lightgbm   : 4.6.0
shap       : 0.49.1


---

## Step 1 — Load & Split

The dataset from N11 is split **temporally** — 2023+2024 for training, 2025 for testing. Shuffling is intentionally avoided: the model must generalize to an unseen season, which mirrors the production use case.

Categorical features (`compound_x`, `compound_y`, `circuit_cluster`) are cast to pandas `category` dtype so LightGBM can handle them natively via its histogram-based categorical splits — no one-hot encoding needed.


In [10]:
# ── Step 1 · Load & split ─────────────────────────────────────────────────────

FEATURES = [
    "gap_ahead_s", "pace_delta_s",
    "tyre_life_x", "tyre_life_y", "tyre_life_diff",
    "speed_trap_delta", "LapNumber", "drs_window",
    "compound_x", "compound_y", "circuit_cluster",
]
CAT_FEATURES = ["compound_x", "compound_y", "circuit_cluster"]
TARGET = "overtake"

In [11]:
def load_dataset(processed_dir):
    df = pd.read_parquet(processed_dir / "overtake_pairs_2023_2025.parquet")
    for col in CAT_FEATURES:
        df[col] = df[col].astype("category")
    return df


def temporal_split(df):
    train = df[df["Year"].isin([2023, 2024])].copy()
    test  = df[df["Year"] == 2025].copy()
    return train, test


def split_xy(df):
    return df[FEATURES], df[TARGET]


def print_split_stats(train, test):
    for name, subset in [("Train (2023+2024)", train), ("Test  (2025)", test)]:
        n = len(subset)
        ot = subset[TARGET].sum()
        print(f"{name}: {n:,} pairs | {ot:,} overtakes ({ot/n*100:.2f}%)")

In [12]:
# ── Run ─────────────────────────────────────────────────────────────────────

df = load_dataset(PROCESSED)
train_df, test_df = temporal_split(df)

X_train, y_train = split_xy(train_df)
X_test,  y_test  = split_xy(test_df)

print_split_stats(train_df, test_df)
print(f"\nFeatures : {FEATURES}")
print(f"Cat cols : {CAT_FEATURES}")

Train (2023+2024): 26,063 pairs | 1,733 overtakes (6.65%)
Test  (2025): 14,105 pairs | 830 overtakes (5.88%)

Features : ['gap_ahead_s', 'pace_delta_s', 'tyre_life_x', 'tyre_life_y', 'tyre_life_diff', 'speed_trap_delta', 'LapNumber', 'drs_window', 'compound_x', 'compound_y', 'circuit_cluster']
Cat cols : ['compound_x', 'compound_y', 'circuit_cluster']


---

## Step 2 — Baseline LightGBM

A first model with near-default parameters to establish a performance floor. `is_unbalance=True` lets LightGBM reweight the positive class automatically (equivalent to `scale_pos_weight ≈ 14.7`). We report both AUC-ROC and AUC-PR — the latter is the primary metric given the class imbalance.


In [ ]:
# ── Step 2 · Baseline LightGBM ────────────────────────────────────────────────


PARAMS_BASE = {
    "objective":     "binary",
    "metric":        "binary_logloss",
    "is_unbalance":  True,
    "n_estimators":  500,
    "learning_rate": 0.05,
    "num_leaves":    31,
    "random_state":  42,
    "n_jobs":        -1,
    "verbose":       -1,
}


def train_lgbm(params, X_tr, y_tr, cat_features=CAT_FEATURES):
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, categorical_feature=cat_features)
    return model


def evaluate(model, X, y, label=""):
    proba = model.predict_proba(X)[:, 1]
    auc_roc = roc_auc_score(y, proba)
    auc_pr  = average_precision_score(y, proba)
    ll      = log_loss(y, proba)
    print(f"{'── ' + label + ' ──':─<40}")
    print(f"  AUC-PR  : {auc_pr:.4f}  (main metric)")
    print(f"  AUC-ROC : {auc_roc:.4f}")
    print(f"  Logloss : {ll:.4f}")
    print(f"\n{classification_report(y, (proba >= 0.5).astype(int), digits=3)}")
    return {"auc_pr": auc_pr, "auc_roc": auc_roc, "logloss": ll}

In [14]:
# ── main ─────────────────────────────────────────────────────────────────────

model_base = train_lgbm(PARAMS_BASE, X_train, y_train)
metrics_base = evaluate(model_base, X_test, y_test, label="Baseline — Test 2025")


AttributeError: module 'lightgbm' has no attribute 'log_loss'